# 📂 Notebook 01 — Data Load and Schema Validation

---

<div style="background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); padding: 24px 28px; border-radius: 16px; margin: 12px 0; border: 1px solid #e94560;">
<h2 style="color: #e94560; margin: 0 0 8px 0; font-size: 1.4em;">📊 Data Load & Schema Validation</h2>
<h3 style="color: #a8dadc; margin: 0 0 16px 0; font-weight: 400; font-size: 1.05em;">Fase A — Fondasi Data dan Eksperimen</h3>
<hr style="border: none; border-top: 1px solid #444; margin: 12px 0;">
<table style="color: #ccc; font-size: 0.9em; border-collapse: collapse; width: 100%;">
<tr><td style="padding: 4px 16px 4px 0; white-space:nowrap;">📍 Studi Kasus</td><td><strong style="color: #fff;">PT. XYZ Banjarmasin, Kalimantan Selatan</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">🎓 Mata Kuliah</td><td><strong style="color: #fff;">Tugas Akhir — ES234733</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">🏛 Institusi</td><td><strong style="color: #fff;">Departemen Sistem Informasi, FTEIC — ITS Surabaya</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">👤 Penulis</td><td><strong style="color: #fff;">Muhammad Iqbal Baiduri Yamani — NRP 5026221103</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">🧑‍🏫 Pembimbing</td><td><strong style="color: #fff;">Edwin Riksakomara, S.Kom., M.T.</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">📅 Tahun</td><td><strong style="color: #fff;">2025</strong></td></tr>
</table>
</div>

## 📌 Tujuan Notebook Ini

Notebook 01 adalah **titik masuk data mentah** ke pipeline. Output notebook ini (DataFrame valid) menjadi input untuk Notebook 02–07.

| # | Tahapan | Keterangan |
|---|---------|------------|
| 1 | **Import & Setup** | Muat library, seed, dan PATHS dari Notebook 00 |
| 2 | **Load Data Excel** | Baca file `.xlsx` menggunakan pandas + openpyxl |
| 3 | **Verifikasi Schema** | Cek kolom wajib: `Year`, `Week`, `Grand Total` |
| 4 | **Ringkasan Dataset** | Shape, tipe data, head/tail, statistik dasar |
| 5 | **Simpan Artefak** | Catat hasil validasi ke `logs/schema_validation.json` |
| 6 | **Checklist** | Verifikasi semua item Pipeline.md terpenuhi |

---

### 🔬 Konteks Data

**Sumber:** Satu file Excel final yang berisi data penjualan mingguan mie instan dari PT. XYZ Banjarmasin.

**Alur Data di Notebook Ini:**

```mermaid
flowchart LR
    A[📁 Dataset_Raw_Top1.xlsx] --> B[Load via pandas]
    B --> C{Kolom Wajib<br/>Year, Week, Grand Total?}
    C -- ❌ Tidak --> D[🛑 Stop: Normalisasi Kolom]
    C -- ✅ Ya --> E[Verifikasi Tipe Data]
    E --> F[Ringkasan Dataset]
    F --> G[💾 logs/schema_validation.json]
    G --> H[✅ Dataset Valid Awal]
    H --> I[➡️ Notebook 02]
```

| Aspek | Detail |
|-------|--------|
| **File Input** | `data/Dataset_Raw_Top 1 Sales 2019 - 2025.xlsx` |
| **Kolom Wajib** | `Year` (int), `Week` (int), `Grand Total` (numeric) |
| **Periode Data** | 2019–2025, frekuensi mingguan |
| **Ekspektasi Baris** | ±364 observasi |
| **Output Notebook** | `df_raw`: DataFrame valid awal |

## ⚙️ 1. Import Library & Setup

In [ ]:
# ── Import Library & Setup (Mirror dari Notebook 00) ──────────
import os
import sys
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ── Global Seed Lock ──────────────────────────────────────────
GLOBAL_SEED = 42
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# ── PATHS ─────────────────────────────────────────────────────
ROOT_DIR = Path(".").resolve().parent
PATHS = {
    "root"         : ROOT_DIR,
    "data"         : ROOT_DIR / "data",
    "logs"         : ROOT_DIR / "logs",
    "outputs"      : ROOT_DIR / "outputs",
    "figures"      : ROOT_DIR / "outputs" / "figures",
    "models"       : ROOT_DIR / "outputs" / "models",
    "metrics"      : ROOT_DIR / "outputs" / "metrics",
    "splits"       : ROOT_DIR / "outputs" / "splits",
    "cv_folds"     : ROOT_DIR / "outputs" / "cv_folds",
    "checkpoints"  : ROOT_DIR / "outputs" / "checkpoints",
    "reports"      : ROOT_DIR / "outputs" / "reports",
    "assets"       : ROOT_DIR / "outputs" / "assets",
    "notebooks"    : ROOT_DIR / "notebook",
    "raw_data_file": ROOT_DIR / "data" / "Dataset_Raw_Top 1 Sales 2019 - 2025.xlsx",
}

print("=" * 60)
print("  SETUP — NOTEBOOK 01")
print("=" * 60)
print(f"\n  Python  : {sys.version.split()[0]}")
print(f"  NumPy   : {np.__version__}")
print(f"  Pandas  : {pd.__version__}")
print(f"  Seed    : {GLOBAL_SEED}")
print(f"  ROOT    : {ROOT_DIR}")
print(f"\n  File target:")
print(f"    {PATHS['raw_data_file'].name}")
exists = PATHS['raw_data_file'].exists()
print(f"    {'✅ File ditemukan' if exists else '❌ File tidak ditemukan'}")
print()

  SETUP — NOTEBOOK 01

  Python  : 3.12.13
  NumPy   : 2.4.4
  Pandas  : 3.0.2
  Seed    : 42
  ROOT    : C:\Users\mikba\Downloads\Semester 7\Pra Tugas Akhir (Retno Aulia Vinarti)\Repository\CNN-BiLSTM-GA-Sales-Forecasting

  File target:
    Dataset_Raw_Top 1 Sales 2019 - 2025.xlsx
    ✅ File ditemukan



## 📂 2. Load Data Excel

In [2]:
# ── Load Data Excel ────────────────────────────────────────────
EXCEL_PATH = PATHS["raw_data_file"]

# ── Deteksi sheet yang tersedia ────────────────────────────────
xl = pd.ExcelFile(EXCEL_PATH, engine="openpyxl")
sheet_names = xl.sheet_names

print("=" * 60)
print("  LOAD DATA EXCEL")
print("=" * 60)
print(f"\n  File  : {EXCEL_PATH.name}")
print(f"  Sheet : {sheet_names}")

# ── Baca sheet pertama ─────────────────────────────────────────
SHEET_USED = sheet_names[0]
df_raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_USED, engine="openpyxl")

print(f"\n  Sheet dipakai : '{SHEET_USED}'")
print(f"  Shape         : {df_raw.shape[0]} baris × {df_raw.shape[1]} kolom")
print(f"  Kolom         : {list(df_raw.columns)}")
print()

  LOAD DATA EXCEL

  File  : Dataset_Raw_Top 1 Sales 2019 - 2025.xlsx
  Sheet : ['Target_Product_Weekly']

  Sheet dipakai : 'Target_Product_Weekly'
  Shape         : 364 baris × 3 kolom
  Kolom         : ['Year', 'Week', 'Grand Total']



## 🔍 3. Verifikasi Schema

In [3]:
# ── Verifikasi Schema ──────────────────────────────────────────
REQUIRED_COLS = ["Year", "Week", "Grand Total"]

print("=" * 60)
print("  VERIFIKASI SCHEMA")
print("=" * 60)

# ── Cek keberadaan kolom wajib ────────────────────────────────
col_checks = {}
for col in REQUIRED_COLS:
    found = col in df_raw.columns
    col_checks[col] = found
    status = "✅" if found else "❌"
    print(f"\n  {status}  Kolom '{col}' {'ditemukan' : <14} → dtype: {df_raw[col].dtype if found else 'N/A'}")

all_cols_ok = all(col_checks.values())
print()
print("=" * 60)
if not all_cols_ok:
    missing = [c for c, ok in col_checks.items() if not ok]
    raise ValueError(f"❌ STOP: Kolom berikut tidak ditemukan: {missing}. Normalisasi kolom dulu!")
print("  ✅ Semua kolom wajib ditemukan.")
print("=" * 60)

# ── Verifikasi tipe data ──────────────────────────────────────
print("\n  Detail dtype kolom wajib:")
print(f"    {'Kolom':<15} {'dtype':<12} {'Non-null':<10} {'Sample'}")
print(f"    {'-'*15} {'-'*12} {'-'*10} {'-'*20}")
for col in REQUIRED_COLS:
    sample_val = df_raw[col].dropna().iloc[0] if df_raw[col].notna().any() else 'N/A'
    print(f"    {col:<15} {str(df_raw[col].dtype):<12} {df_raw[col].notna().sum():<10} {sample_val}")
print()

  VERIFIKASI SCHEMA

  ✅  Kolom 'Year' ditemukan      → dtype: int64

  ✅  Kolom 'Week' ditemukan      → dtype: int64

  ✅  Kolom 'Grand Total' ditemukan      → dtype: int64

  ✅ Semua kolom wajib ditemukan.

  Detail dtype kolom wajib:
    Kolom           dtype        Non-null   Sample
    --------------- ------------ ---------- --------------------
    Year            int64        364        2019
    Week            int64        364        1
    Grand Total     int64        364        53605



In [4]:
# ── Cek Missing Value pada Kolom Wajib ────────────────────────
print("=" * 60)
print("  CEK MISSING VALUE (KOLOM WAJIB)")
print("=" * 60)

missing_summary = {}
for col in REQUIRED_COLS:
    n_missing = df_raw[col].isna().sum()
    pct = n_missing / len(df_raw) * 100
    missing_summary[col] = {"count": int(n_missing), "pct": round(pct, 2)}
    status = "✅" if n_missing == 0 else "⚠️"
    print(f"\n  {status}  '{col}': {n_missing} missing ({pct:.1f}%)")

print()
print("=" * 60)
total_missing = sum(v["count"] for v in missing_summary.values())
if total_missing == 0:
    print("  ✅ Tidak ada missing value pada kolom wajib.")
else:
    print(f"  ⚠️  Total {total_missing} missing value ditemukan. Lihat Notebook 02.")
print("=" * 60)
print()

  CEK MISSING VALUE (KOLOM WAJIB)

  ✅  'Year': 0 missing (0.0%)

  ✅  'Week': 0 missing (0.0%)

  ✅  'Grand Total': 0 missing (0.0%)

  ✅ Tidak ada missing value pada kolom wajib.



## 📊 4. Ringkasan Dataset

In [5]:
# ── Ringkasan Dataset ──────────────────────────────────────────
print("=" * 60)
print("  RINGKASAN DATASET")
print("=" * 60)

print(f"\n  Shape          : {df_raw.shape[0]} baris × {df_raw.shape[1]} kolom")
print(f"  Kolom          : {list(df_raw.columns)}")
print(f"  Year range     : {int(df_raw['Year'].min())} – {int(df_raw['Year'].max())}")
print(f"  Week range     : {int(df_raw['Week'].min())} – {int(df_raw['Week'].max())}")
print(f"  Grand Total    : min={df_raw['Grand Total'].min():,.0f}  max={df_raw['Grand Total'].max():,.0f}  mean={df_raw['Grand Total'].mean():,.1f}")

print("\n  --- 5 Baris Pertama ---")
print(df_raw[["Year", "Week", "Grand Total"]].head().to_string(index=True))

print("\n  --- 5 Baris Terakhir ---")
print(df_raw[["Year", "Week", "Grand Total"]].tail().to_string(index=True))

print("\n  --- Statistik Deskriptif Grand Total ---")
print(df_raw[["Grand Total"]].describe().to_string())
print()

  RINGKASAN DATASET

  Shape          : 364 baris × 3 kolom
  Kolom          : ['Year', 'Week', 'Grand Total']
  Year range     : 2019 – 2025
  Week range     : 1 – 52
  Grand Total    : min=1,305  max=112,040  mean=53,322.9

  --- 5 Baris Pertama ---
   Year  Week  Grand Total
0  2019     1        53605
1  2019     2        58298
2  2019     3        67320
3  2019     4        28879
4  2019     5        45030

  --- 5 Baris Terakhir ---
     Year  Week  Grand Total
359  2025    48        51166
360  2025    49        32576
361  2025    50        60694
362  2025    51        24018
363  2025    52        56965

  --- Statistik Deskriptif Grand Total ---
         Grand Total
count     364.000000
mean    53322.895604
std     18426.244454
min      1305.000000
25%     40078.750000
50%     51108.500000
75%     65113.500000
max    112040.000000



## 💾 5. Simpan Artefak Validasi

In [6]:
# ── Simpan Schema Validation Log ──────────────────────────────
schema_log = {
    "notebook"       : "01 - Data Load and Schema Validation",
    "timestamp"      : datetime.now().isoformat(),
    "file"           : str(PATHS["raw_data_file"]),
    "sheet_used"     : SHEET_USED,
    "shape"          : {"rows": df_raw.shape[0], "cols": df_raw.shape[1]},
    "columns"        : list(df_raw.columns),
    "required_cols"  : REQUIRED_COLS,
    "col_checks"     : col_checks,
    "missing_values" : missing_summary,
    "year_range"     : [int(df_raw["Year"].min()), int(df_raw["Year"].max())],
    "week_range"     : [int(df_raw["Week"].min()), int(df_raw["Week"].max())],
    "grand_total_stats": {
        "min"  : round(float(df_raw["Grand Total"].min()), 2),
        "max"  : round(float(df_raw["Grand Total"].max()), 2),
        "mean" : round(float(df_raw["Grand Total"].mean()), 2),
        "std"  : round(float(df_raw["Grand Total"].std()), 2),
    },
    "schema_status"  : "VALID" if all_cols_ok else "INVALID",
}

log_path = PATHS["logs"] / "schema_validation.json"
log_path.parent.mkdir(parents=True, exist_ok=True)
with open(log_path, "w") as f:
    json.dump(schema_log, f, indent=2)

print("=" * 60)
print("  ARTEFAK VALIDASI TERSIMPAN")
print("=" * 60)
print(f"\n  ✅  {log_path}")
print(f"      Status schema : {schema_log['schema_status']}")
print(f"      Timestamp     : {schema_log['timestamp']}")
print()

  ARTEFAK VALIDASI TERSIMPAN

  ✅  C:\Users\mikba\Downloads\Semester 7\Pra Tugas Akhir (Retno Aulia Vinarti)\Repository\CNN-BiLSTM-GA-Sales-Forecasting\logs\schema_validation.json
      Status schema : VALID
      Timestamp     : 2026-05-05T20:24:15.770319



## ✅ 6. Checklist

> Verifikasi bahwa semua persyaratan Notebook 01 dari Pipeline.md telah dipenuhi  
> sebelum melanjutkan ke Notebook 02.

| # | Item | Status |
|---|------|--------|
| 1 | Kolom `Year` ada | ✅ `int64`, range 2019–2025 |
| 2 | Kolom `Week` ada | ✅ `int64`, range 1–53 |
| 3 | Kolom `Grand Total` ada | ✅ `int64`, numeric |
| 4 | Jumlah baris terbaca dengan benar | ✅ 364 baris × 3 kolom |

In [7]:
# ── Checklist Notebook 01 ──────────────────────────────────────
checklist = [
    (
        "Kolom Year ada",
        f"dtype={df_raw['Year'].dtype}, range={int(df_raw['Year'].min())}–{int(df_raw['Year'].max())}",
        col_checks.get("Year", False),
    ),
    (
        "Kolom Week ada",
        f"dtype={df_raw['Week'].dtype}, range={int(df_raw['Week'].min())}–{int(df_raw['Week'].max())}",
        col_checks.get("Week", False),
    ),
    (
        "Kolom Grand Total ada",
        f"dtype={df_raw['Grand Total'].dtype}, mean={df_raw['Grand Total'].mean():,.1f}",
        col_checks.get("Grand Total", False),
    ),
    (
        "Jumlah baris terbaca dengan benar",
        f"{df_raw.shape[0]} baris × {df_raw.shape[1]} kolom (ekspektasi ±364)",
        df_raw.shape[0] > 0,
    ),
]

print("=" * 60)
print("  CHECKLIST — NOTEBOOK 01")
print("=" * 60)

all_pass = True
for item, detail, passed in checklist:
    icon = "✅" if passed else "❌"
    if not passed:
        all_pass = False
    print(f"\n  {icon}  {item}")
    print(f"      → {detail}")

print()
print("=" * 60)
status = "✅ SEMUA ITEM SELESAI — NOTEBOOK 01 COMPLETE" if all_pass else "❌ ADA ITEM BELUM SELESAI"
print(f"  STATUS   : {status}")
print("=" * 60)
print()
print("  🚀 Lanjut ke:")
print("     📓 Notebook 02 — Time Index Integrity Audit")
print()

  CHECKLIST — NOTEBOOK 01

  ✅  Kolom Year ada
      → dtype=int64, range=2019–2025

  ✅  Kolom Week ada
      → dtype=int64, range=1–52

  ✅  Kolom Grand Total ada
      → dtype=int64, mean=53,322.9

  ✅  Jumlah baris terbaca dengan benar
      → 364 baris × 3 kolom (ekspektasi ±364)

  STATUS   : ✅ SEMUA ITEM SELESAI — NOTEBOOK 01 COMPLETE

  🚀 Lanjut ke:
     📓 Notebook 02 — Time Index Integrity Audit



---

## 🔗 Navigasi Pipeline

| | Notebook |
|--|----------|
| ← | **[00 - Project Scope and Reproducibility](./00%20-%20Project%20Scope%20and%20Reproducibility.ipynb)** |
| **→** | **[02 - Time Index Integrity Audit](./02%20-%20Time%20Index%20Integrity%20Audit.ipynb)** |

---

### 📎 Variabel Penting yang Dihasilkan Notebook Ini

Variabel berikut tersedia untuk notebook selanjutnya (dimuat ulang dari file atau dihitung kembali):

```python
df_raw        # DataFrame mentah hasil load Excel — shape (~364, 3)
              # Kolom: Year (int), Week (int), Grand Total (float)
SHEET_USED    # Nama sheet Excel yang dipakai
PATHS         # dict path pipeline (sama seperti Notebook 00)
```

---

<div style="text-align: center; color: #888; font-size: 0.85em; padding: 12px 0;">
Notebook 01 — Data Load &amp; Schema Validation &nbsp;|&nbsp;
CNN–BiLSTM + GA Sales Forecasting &nbsp;|&nbsp;
Departemen Sistem Informasi ITS Surabaya &nbsp;|&nbsp; 2025
</div>